# Robinhood Chain Phase 0 token-market data-quality audit

## tl;dr

**Assessment: FAIL for the requested 3-token × 20-open trade-level audit.** Canonical contracts, verified spot pools, live swaps, and venue-specific price reconstruction were all confirmed. The blocking issue is historical data access: the official public RPC returned rate limits, timeouts, and a 10,000-log result cap; Blockscout's API timed out; and the fallback RPC did not complete the high-volume history reliably within the bounded audit.

The 60-row table preserves every requested token-session. It contains 28 exact pre-pool-creation zero rows and marks the remaining 32 rows incomplete rather than inventing trade-level metrics. GeckoTerminal hourly OHLCV supplies a coverage lower bound only; it is not treated as raw fills.


## Context & methods

- Research question remains unchanged: can Robinhood Chain Stock Token overnight trading contain information about the next US equity market open?
- This artifact is only Phase 0 data feasibility validation; it performs no predictive test.
- Exactly three canonical Stock Token/USDG pools were selected from a bounded current top-pool scan and verified against recent onchain swap events:
  - AMC — Uniswap V4 pool `0x7499938c352d5b5b8f0c648722aca5ee964ef9b85c3a3041f1ec379726291d9d`
  - NVDA — Uniswap V3 pool `0xd4eb21209c4d6093f80b5b84f5c45cc093ea14a3`
  - HIMS — Uniswap V3 pool `0xc8c90d3a1c1a24967e773ac2ad0d456ba3e31f64`
- The sample is exactly 20 completed NYSE opens from 2026-08-10 through 2026-09-04.
- Each observation window runs from the previous valid NYSE core close through the next 09:30 ET open. America/New_York timestamps and the official 2026 NYSE holiday calendar preserve weekends, holidays, DST rules, and early-close logic.

Primary references: [Stock Token API](https://docs.robinhood.com/chain/stock-token-apis/), [Robinhood Chain RPC](https://docs.robinhood.com/chain/connecting/), [NYSE calendar](https://www.nyse.com/trade/hours-calendars), [Uniswap venues](https://docs.robinhood.com/chain/building-with-stock-tokens/), and [GeckoTerminal API limitations](https://api.geckoterminal.com/docs/index.html).


## Data

The machine-readable table has one row per token × next market open. Blank trade-level fields mean **not verified**, not zero. Only rows labeled `complete_pre_creation_zero` are exact zeros.


In [1]:
import json
from pathlib import Path

import pandas as pd

project_root = Path.cwd() if (Path.cwd() / "reports").exists() else Path.cwd().parent
table_path = project_root / "reports/robinhood_chain_phase0/token_session_quality.csv"
summary_path = project_root / "reports/robinhood_chain_phase0/audit_summary.json"

audit = pd.read_csv(table_path)
summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(audit.shape)
display(audit.head(6))


(60, 27)
  token_symbol                    canonical_token_contract       venue                                                  pool_id_or_address   pool_created_at_utc quote_symbol                              quote_contract market_open_date            window_start_et              window_end_et  window_hours  swap_fill_count  active_trading_minutes  quote_notional_usdg  median_trade_size_usdg  unique_event_sender_addresses  first_trade_timestamp_et  last_trade_timestamp_et  max_gap_minutes  price_reconstruction_rate  indexer_full_hours_with_volume  indexer_full_hour_volume_usd  indexer_nonempty_lower_bound rpc_indexer_status  pool_mapping_verified zero_trade_session           collection_status
0          AMC  0x05a3d1cd21d0c88145e82600e62e7e496e0f222b  Uniswap V4  0x7499938c352d5b5b8f0c648722aca5ee964ef9b85c3a3041f1ec379726291d9d  2026-09-04T04:33:55Z         USDG  0x5fc5360d0400a0fd4f2af552add042d716f1d168       2026-08-10  2026-08-07T16:00:00-04:00  2026-08-10T09:30:00-04:00       

## Results

### Coverage and completeness


In [2]:
coverage = audit.groupby("token_symbol").agg(
    requested_sessions=("market_open_date", "size"),
    exact_pre_creation_zeros=("collection_status", lambda values: (values == "complete_pre_creation_zero").sum()),
    indexer_nonempty_session_lower_bound=("indexer_nonempty_lower_bound", "sum"),
    indexed_full_hour_volume_usd=("indexer_full_hour_volume_usd", "sum"),
)
display(coverage)
print("Duplicate token-session keys:", audit.duplicated(["token_symbol", "market_open_date"]).sum())
print("Date range:", audit["market_open_date"].min(), "to", audit["market_open_date"].max())


              requested_sessions  exact_pre_creation_zeros  indexer_nonempty_session_lower_bound  indexed_full_hour_volume_usd
token_symbol                                                                                                                  
AMC                           20                        19                                     1                  1.762886e+07
HIMS                          20                         9                                     8                  2.083678e+07
NVDA                          20                         0                                    20                  2.839818e+08
Duplicate token-session keys: 0
Date range: 2026-08-10 to 2026-09-04


### Live execution and reconstruction verification

Small recent RPC probes confirmed the selected contracts emit the correct Uniswap swap events and that signed amount deltas reconstruct plausible USDG execution prices:

| Token | Recent bounded swap count | Sample reconstructed price | Sample quote size |
|---|---:|---:|---:|
| AMC | 10 | 2.6233 USDG | 2,359.94 USDG |
| NVDA | 81 | 232.1801 USDG | 24.66 USDG |
| HIMS | 2 | 27.9839 USDG | 197.10 USDG |

Across one common 500-block range, official Robinhood RPC and the fallback provider returned identical counts: AMC 7, NVDA 39, HIMS 2. This verifies recent-provider consistency only, not the 20-session history. Blockscout reconciliation was unavailable because its API timed out.

### Required metrics that remain unavailable

For the 32 rows during a pool's active lifespan, raw swap count, exact active minutes, exact quote notional, median trade size, participant count, exact first/last fills, maximum gap, and reconstruction rate are intentionally blank. GeckoTerminal hourly candles cannot validly supply those trade-level quantities.


## Takeaways

**A. Selected tokens/pools:** AMC, NVDA, and HIMS; these were the three highest-current-volume canonical Stock Token/USDG pools in the bounded top-pool scan and all emitted reconstructable recent spot swaps.

**B. Nonempty sessions:** Exact trade-level counts are not available. Hourly-indexer lower bounds are AMC 1/20, NVDA 20/20, and HIMS 8/20. AMC's pool did not exist for 19 sessions; HIMS did not exist for 9.

**C. Activity:** Indexed full-hour volume is material where present, especially NVDA, but cannot be reconciled to complete raw executions under the available access constraints.

**D. Reconstructed prices:** Recent events reconstruct correctly. Consistency across all 20 sessions is unverified.

**E. Participants and gaps:** Not assessable from hourly OHLCV. Uniswap event `sender` is frequently a router/hook rather than the economic end-user, so even raw unique-sender counts require careful interpretation.

**F. RPC/indexer consistency:** Recent official-RPC versus fallback-RPC counts matched in a 500-block probe. Full Blockscout reconciliation failed because the API was unreachable from the audit environment.

**G. Ambiguities:** Canonical token and pool mapping are verified; event ABIs and token/quote ordering are resolved. Historical access, complete trade-level coverage, end-user identity, and indexer reconciliation remain material gaps. No selected-token corporate action was effective during the sample, so the multiplier was 1.0 where checked.

**H. Final assessment — FAIL.** The requested evidence cannot meet its own decision rule without a dependable archive/indexed trade source. The minimum unblock is credentials for a documented archive RPC or a queryable complete Uniswap indexer, followed by rerunning only these 3 pools and 20 sessions.
